In [4]:
# ============================================================
# CELL 1 — SETUP, MOUNT DRIVE, EXTRACT LoRA ADAPTER,
# LOAD SAVED RAG ARTIFACTS AND FIXED TEST DATA
#
# This cell:
# 1. Installs the required libraries
# 2. Mounts Google Drive only when necessary
# 3. Finds and safely extracts the fine-tuned model ZIP
# 4. Locates adapter_config.json even inside nested folders
# 5. Loads FAISS, BM25, saved KB chunks and RAG configuration
# 6. Loads only the four test files
#
# The uploaded *_qa_train.json files are NOT used.
# ============================================================


# ------------------------------------------------------------
# 1. Install required libraries
# ------------------------------------------------------------

%pip install -q -U \
    "transformers>=4.49,<5" \
    "peft>=0.14,<1" \
    "accelerate>=1.2" \
    "bitsandbytes>=0.45" \
    "sentence-transformers>=3.4" \
    faiss-cpu \
    rank-bm25 \
    rapidfuzz \
    sacrebleu \
    bert-score==0.3.13 \
    nltk


# ------------------------------------------------------------
# 2. Import libraries
# ------------------------------------------------------------

from google.colab import drive

import gc
import json
import os
import pickle
import random
import re
import shutil
import time
import unicodedata
import zipfile

from collections import Counter, defaultdict
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch

# Import this before loading the saved BM25 pickle.
from rank_bm25 import BM25Okapi

from IPython.display import display
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 3. Mount Google Drive
# ------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive")

if not (DRIVE_ROOT / "MyDrive").exists():

    drive.mount(
        "/content/drive",
        force_remount=False
    )

else:

    print(
        "Google Drive is already mounted."
    )


# ------------------------------------------------------------
# 4. Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)


print(
    "Runtime device:",
    "GPU" if torch.cuda.is_available() else "CPU"
)


# ------------------------------------------------------------
# 5. Main paths
# ------------------------------------------------------------

BASE_DIR = Path("/content")

ART_DIR = Path(
    "/content/drive/MyDrive/govt_rag_artifacts"
)

ADAPTER_EXTRACT_ROOT = Path(
    "/content/Govt_chatbot_finetuned_model_extracted"
)


# ------------------------------------------------------------
# 6. Automatically locate the uploaded adapter ZIP
# ------------------------------------------------------------

expected_zip_names = [

    "Govt_chatbot_finetuned_model.zip",
    "Govt_Chatbot_finetuned_model.zip",
    "govt_chatbot_finetuned_model.zip"
]


ADAPTER_ZIP = None


# First try expected names.
for zip_name in expected_zip_names:

    candidate_path = BASE_DIR / zip_name

    if candidate_path.exists():

        ADAPTER_ZIP = candidate_path
        break


# Otherwise search all ZIP files in /content.
if ADAPTER_ZIP is None:

    available_zip_files = list(
        BASE_DIR.glob("*.zip")
    )

    matching_zip_files = [

        path

        for path in available_zip_files

        if (
            "govt" in path.name.lower()
            and "finetun" in path.name.lower()
        )
    ]


    if matching_zip_files:

        ADAPTER_ZIP = matching_zip_files[0]


if ADAPTER_ZIP is None:

    available_zip_names = [

        path.name

        for path in BASE_DIR.glob("*.zip")
    ]

    raise FileNotFoundError(

        "The fine-tuned model ZIP could not be found in /content.\n\n"

        f"ZIP files currently available: {available_zip_names}\n\n"

        "Upload Govt_chatbot_finetuned_model.zip and run Cell 1 again."
    )


print("\nFine-tuned model ZIP found:")
print(ADAPTER_ZIP)


# ------------------------------------------------------------
# 7. Validate the ZIP file
# ------------------------------------------------------------

if not zipfile.is_zipfile(ADAPTER_ZIP):

    raise zipfile.BadZipFile(

        f"This is not a valid ZIP file:\n{ADAPTER_ZIP}"
    )


with zipfile.ZipFile(
    ADAPTER_ZIP,
    "r"
) as zip_file:

    zip_members = zip_file.namelist()


print("\nFiles detected inside the ZIP:")

for member in zip_members:

    print("-", member)


if not any(

    Path(member).name
    == "adapter_config.json"

    for member in zip_members
):

    raise FileNotFoundError(

        "adapter_config.json is not present inside the ZIP.\n"

        "This ZIP therefore does not appear to contain a PEFT/LoRA adapter."
    )


if not any(

    Path(member).name
    in {
        "adapter_model.safetensors",
        "adapter_model.bin"
    }

    for member in zip_members
):

    raise FileNotFoundError(

        "No adapter weight file was found inside the ZIP.\n"

        "Expected adapter_model.safetensors or adapter_model.bin."
    )


# ------------------------------------------------------------
# 8. Remove any incomplete old extraction
# ------------------------------------------------------------

if ADAPTER_EXTRACT_ROOT.exists():

    print(
        "\nRemoving the previous extracted adapter folder..."
    )

    shutil.rmtree(
        ADAPTER_EXTRACT_ROOT
    )


ADAPTER_EXTRACT_ROOT.mkdir(

    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 9. Extract the adapter again
# ------------------------------------------------------------

print(
    "\nExtracting the fine-tuned LoRA adapter..."
)


with zipfile.ZipFile(
    ADAPTER_ZIP,
    "r"
) as zip_file:

    zip_file.extractall(
        ADAPTER_EXTRACT_ROOT
    )


# ------------------------------------------------------------
# 10. Locate adapter_config.json recursively
# ------------------------------------------------------------

adapter_config_paths = list(

    ADAPTER_EXTRACT_ROOT.rglob(
        "adapter_config.json"
    )
)


if not adapter_config_paths:

    extracted_files = [

        str(path.relative_to(
            ADAPTER_EXTRACT_ROOT
        ))

        for path in ADAPTER_EXTRACT_ROOT.rglob("*")

        if path.is_file()
    ]

    raise FileNotFoundError(

        "The ZIP was extracted, but adapter_config.json "
        "could not be located.\n\n"

        f"Extracted files: {extracted_files}"
    )


if len(adapter_config_paths) > 1:

    print(

        "\nWarning: More than one adapter_config.json was found."

    )

    for path in adapter_config_paths:

        print("-", path)


ADAPTER_DIR = adapter_config_paths[0].parent


print(
    "\nLoRA adapter extracted successfully."
)

print(
    "Adapter directory:",
    ADAPTER_DIR
)


print("\nAdapter files:")

for path in sorted(
    ADAPTER_DIR.iterdir()
):

    if path.is_file():

        print(
            f"- {path.name} "
            f"({path.stat().st_size / 1024:.1f} KB)"
        )


# ------------------------------------------------------------
# 11. Verify essential adapter files
# ------------------------------------------------------------

ADAPTER_CONFIG_FILE = (
    ADAPTER_DIR / "adapter_config.json"
)


adapter_weight_candidates = [

    ADAPTER_DIR / "adapter_model.safetensors",
    ADAPTER_DIR / "adapter_model.bin"
]


ADAPTER_WEIGHT_FILE = next(

    (
        path

        for path in adapter_weight_candidates

        if path.exists()
    ),

    None
)


if not ADAPTER_CONFIG_FILE.exists():

    raise FileNotFoundError(

        f"Missing adapter configuration:\n"
        f"{ADAPTER_CONFIG_FILE}"
    )


if ADAPTER_WEIGHT_FILE is None:

    raise FileNotFoundError(

        "The adapter configuration exists, but no adapter "
        "weight file was found."
    )


with open(
    ADAPTER_CONFIG_FILE,
    "r",
    encoding="utf-8"
) as file:

    adapter_config_preview = json.load(file)


print("\nAdapter verification:")

print(
    "Base model:",
    adapter_config_preview.get(
        "base_model_name_or_path",
        "Not recorded"
    )
)

print(
    "PEFT type:",
    adapter_config_preview.get(
        "peft_type",
        "Not recorded"
    )
)

print(
    "LoRA rank:",
    adapter_config_preview.get(
        "r",
        "Not recorded"
    )
)

print(
    "Adapter weights:",
    ADAPTER_WEIGHT_FILE.name
)


# ------------------------------------------------------------
# 12. Define fixed test file paths
# ------------------------------------------------------------

TEST_FILES = {

    "passport":
        BASE_DIR / "passport_qa_test.json",

    "birth_death":
        BASE_DIR / "birth_death_qa_test.json",

    "nid":
        BASE_DIR / "nid_qa_test.json",

    "tin":
        BASE_DIR / "TIN_qa_test.json"
}


# ------------------------------------------------------------
# 13. Define saved RAG artifact paths
# ------------------------------------------------------------

ARTIFACT_FILES = {

    "config":
        ART_DIR / "rag_config.json",

    "chunks":
        ART_DIR / "merged_kb_chunks.json",

    "faiss":
        ART_DIR / "bge_m3_faiss.index",

    "embeddings":
        ART_DIR / "bge_m3_embeddings.npy",

    "bm25":
        ART_DIR / "bm25_index.pkl"
}


# ------------------------------------------------------------
# 14. Check all required files
# ------------------------------------------------------------

missing_files = []


for domain, path in TEST_FILES.items():

    if not path.exists():

        missing_files.append(
            f"Test file ({domain}): {path}"
        )


for artifact_name in [

    "config",
    "chunks",
    "faiss",
    "bm25"

]:

    path = ARTIFACT_FILES[
        artifact_name
    ]

    if not path.exists():

        missing_files.append(
            f"RAG artifact ({artifact_name}): {path}"
        )


if missing_files:

    raise FileNotFoundError(

        "The following required files are missing:\n\n- "

        + "\n- ".join(missing_files)
    )


print(
    "\nAll required test files and RAG artifacts were found."
)


# ------------------------------------------------------------
# 15. General helper functions
# ------------------------------------------------------------

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:

        return json.load(file)


def extract_records(
    data,
    file_name
):

    if isinstance(data, list):

        return data


    if isinstance(data, dict):

        for key in [

            "data",
            "records",
            "items",
            "questions",
            "qa_pairs"

        ]:

            if isinstance(
                data.get(key),
                list
            ):

                return data[key]


    raise ValueError(

        f"Could not find a JSON record list in {file_name}."
    )


def normalize_id(value):

    value = str(
        value
    ).strip()


    if re.fullmatch(
        r"\d+\.0",
        value
    ):

        value = value[:-2]


    return value


def normalize_domain(domain):

    domain = str(
        domain
    ).strip().lower()


    if "passport" in domain:

        return "passport"


    if (
        "birth" in domain
        or "death" in domain
    ):

        return "birth_death"


    if "nid" in domain:

        return "nid"


    if (
        "tin" in domain
        or "tax" in domain
    ):

        return "tin"


    return domain


def tokenize_bn_en(text):

    return re.findall(

        r"[\u0980-\u09FF]+|[A-Za-z0-9]+",

        str(text).lower()
    )


def get_chunk_text(chunk):

    return str(

        chunk.get("chunk_text")
        or chunk.get("text")
        or chunk.get("content")
        or chunk.get("passage")
        or ""

    ).strip()


# ------------------------------------------------------------
# 16. Load saved RAG configuration
# ------------------------------------------------------------

rag_config = load_json(

    ARTIFACT_FILES["config"]
)


print("\nRAG configuration loaded:")

print(
    json.dumps(
        rag_config,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# 17. Load saved KB chunks
# ------------------------------------------------------------

chunk_data = load_json(

    ARTIFACT_FILES["chunks"]
)


if isinstance(
    chunk_data,
    list
):

    chunks = chunk_data


elif (
    isinstance(chunk_data, dict)
    and isinstance(
        chunk_data.get("chunks"),
        list
    )
):

    chunks = chunk_data["chunks"]


else:

    raise ValueError(

        "merged_kb_chunks.json must contain either:\n"

        "1. A JSON list of chunks, or\n"

        "2. An object containing a 'chunks' list."
    )


if len(chunks) == 0:

    raise ValueError(

        "The saved KB chunk file is empty."
    )


# Add a fallback chunk ID when one was not saved.
for chunk_index, chunk in enumerate(chunks):

    if not isinstance(
        chunk,
        dict
    ):

        raise ValueError(

            f"Chunk {chunk_index} is not a JSON object."
        )


    if not chunk.get("chunk_id"):

        chunk["chunk_id"] = (
            f"chunk_{chunk_index:04d}"
        )


    chunk["domain"] = normalize_domain(

        chunk.get(
            "domain",
            ""
        )
    )


# ------------------------------------------------------------
# 18. Load saved FAISS index
# ------------------------------------------------------------

faiss_index = faiss.read_index(

    str(
        ARTIFACT_FILES["faiss"]
    )
)


# ------------------------------------------------------------
# 19. Load saved BM25 index
# ------------------------------------------------------------

with open(

    ARTIFACT_FILES["bm25"],
    "rb"

) as file:

    bm25 = pickle.load(file)


# ------------------------------------------------------------
# 20. Validate index and chunk alignment
# ------------------------------------------------------------

if faiss_index.ntotal != len(chunks):

    raise ValueError(

        "FAISS/chunk mismatch detected.\n"

        f"FAISS vectors: {faiss_index.ntotal}\n"

        f"Saved chunks: {len(chunks)}\n\n"

        "The index and merged_kb_chunks.json must come "
        "from the same RAG run."
    )


try:

    bm25_document_count = len(
        bm25.doc_freqs
    )

except Exception:

    bm25_document_count = None


if (
    bm25_document_count is not None
    and bm25_document_count != len(chunks)
):

    raise ValueError(

        "BM25/chunk mismatch detected.\n"

        f"BM25 documents: {bm25_document_count}\n"

        f"Saved chunks: {len(chunks)}"
    )


EMBED_MODEL_NAME = (

    rag_config.get(
        "embedding_model"
    )

    or rag_config.get(
        "embed_model"
    )

    or "BAAI/bge-m3"
)


print(
    "\nSaved RAG artifacts loaded successfully."
)

print(
    "Embedding model:",
    EMBED_MODEL_NAME
)

print(
    "Total KB chunks:",
    len(chunks)
)

print(
    "FAISS vectors:",
    faiss_index.ntotal
)

if bm25_document_count is not None:

    print(
        "BM25 documents:",
        bm25_document_count
    )


# ------------------------------------------------------------
# 21. Load the four fixed test sets
# ------------------------------------------------------------

def load_test_rows():

    rows = []


    for domain, path in TEST_FILES.items():

        test_data = load_json(
            path
        )


        records = extract_records(

            test_data,
            path.name
        )


        for row_index, item in enumerate(
            records
        ):

            if not isinstance(
                item,
                dict
            ):

                raise ValueError(

                    f"Record {row_index} in {path.name} "
                    "is not a JSON object."
                )


            question = (

                item.get("instruction")

                or item.get("question")

                or item.get("query")

                or item.get("input")

                or ""
            )


            gold_answer = (

                item.get("output")

                or item.get("answer")

                or item.get("response")

                or item.get("gold")

                or item.get("target")

                or ""
            )


            record_id = normalize_id(

                item.get(
                    "id",
                    f"{domain}_{row_index + 1}"
                )
            )


            if not str(
                question
            ).strip():

                raise ValueError(

                    f"Missing question in {path.name}, "
                    f"record {row_index}."
                )


            if not str(
                gold_answer
            ).strip():

                raise ValueError(

                    f"Missing gold answer in {path.name}, "
                    f"record {row_index}."
                )


            rows.append({

                "domain":
                    domain,

                "id":
                    record_id,

                "question":
                    str(question).strip(),

                "gold_answer":
                    str(gold_answer).strip(),

                "gold_source_url":
                    str(
                        item.get(
                            "source_url",
                            ""
                        )
                    ).strip()
            })


    return rows


test_rows = load_test_rows()


if len(test_rows) == 0:

    raise ValueError(

        "No test questions were loaded."
    )


test_df = pd.DataFrame(
    test_rows
)


# ------------------------------------------------------------
# 22. Final Cell 1 summary
# ------------------------------------------------------------

print(
    "\n========================================"
)

print(
    "CELL 1 COMPLETED SUCCESSFULLY"
)

print(
    "========================================"
)

print(
    "Adapter directory:",
    ADAPTER_DIR
)

print(
    "RAG artifact directory:",
    ART_DIR
)

print(
    "Total test questions:",
    len(test_rows)
)

print(
    "Training files used:",
    "No"
)


print(
    "\nTest questions by domain:"
)


display(

    test_df

    .groupby(
        "domain"
    )

    .size()

    .rename(
        "count"
    )

    .reset_index()
)


print(
    "\nTest data preview:"
)


display(

    test_df[
        [
            "id",
            "domain",
            "question",
            "gold_answer"
        ]
    ].head(3)
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 40.2 MB/s eta 0:00:00
Google Drive is already mounted.
Runtime device: GPU

Fine-tuned model ZIP found:
/content/Govt_chatbot_finetuned_model.zip

Files detected inside the ZIP:
- tokenizer.json
- README.md
- adapter_model.safetensors
- tokenizer_config.json
- chat_template.jinja
- adapter_config.json

Removing the previous extracted adapter folder...

Extracting the fine-tuned LoRA adapter...

LoRA adapter extracted successfully.
Adapter directory: /content/Govt_chatbot_finetuned_model_extracted

Adapter files:
- README.md (5.1 KB)
- adapter_config.json (1.2 KB)
- adapter_model.safetensors (157747.3 KB)
- chat_template.jinja (2.4 KB)
- tokenizer.json (11154.5 KB)
- tokenizer_config.json (4.3 KB)

Adapter verification:
Base model: unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit
PEFT type: LORA
LoRA rank: 16
Adapter weights: adapter_model.safetensors

All required test files and RAG artifacts were found.

RAG configuration loaded:


,domain,count
0,birth_death,63
1,nid,74
2,passport,63
3,tin,48



Test data preview:


,id,domain,question,gold_answer
0,passport_001,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...
1,passport_179,passport,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...
2,passport_216,passport,বিদেশে শ্রমিক শিক্ষার্থীর ৬৪ পৃষ্ঠা ৫ বছর এক্স...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...


In [5]:
# ============================================================
# CELL 2 — HYBRID RETRIEVAL AND CROSS-ENCODER RERANKING
# Performs:
# 1. BGE-M3 dense retrieval
# 2. BM25 lexical retrieval
# 3. Reciprocal Rank Fusion
# 4. BGE multilingual reranking
# 5. Returns the final top-5 KB chunks
# ============================================================

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer
)


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ------------------------------------------------------------
# Retrieval settings
# ------------------------------------------------------------

RERANKER_MODEL_NAME = (
    "BAAI/bge-reranker-v2-m3"
)

DENSE_K = 80
BM25_K = 80

RERANK_TOP_N = 20
FINAL_K = 5

RRF_CONSTANT = 60
DOMAIN_BOOST = 0.025


# True keeps the same domain-aware setting as the old RAG run.
# For real-world testing, this should later be changed to False.
USE_ORACLE_DOMAIN_FILTER = True


# ------------------------------------------------------------
# Load BGE-M3 embedding model
# ------------------------------------------------------------

print(
    "Loading embedding model:",
    EMBED_MODEL_NAME
)

embedder = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE
)


# ------------------------------------------------------------
# Load multilingual reranker
# ------------------------------------------------------------

print(
    "Loading reranker:",
    RERANKER_MODEL_NAME
)

reranker_tokenizer = (
    AutoTokenizer.from_pretrained(
        RERANKER_MODEL_NAME
    )
)

reranker_model = (
    AutoModelForSequenceClassification
    .from_pretrained(

        RERANKER_MODEL_NAME,

        torch_dtype=(
            torch.float16
            if DEVICE == "cuda"
            else torch.float32
        )
    )
)

reranker_model.to(DEVICE)
reranker_model.eval()


# ------------------------------------------------------------
# Cross-encoder reranking
# ------------------------------------------------------------

def rerank_candidates(
    question,
    candidates,
    final_k=FINAL_K
):

    if not candidates:
        return []

    questions = [
        question
    ] * len(candidates)

    passages = [
        get_chunk_text(candidate)
        for candidate in candidates
    ]


    encoded = reranker_tokenizer(

        questions,
        passages,

        padding=True,
        truncation=True,

        max_length=512,

        return_tensors="pt"
    )


    encoded = {

        key: value.to(DEVICE)

        for key, value
        in encoded.items()
    }


    with torch.inference_mode():

        logits = (
            reranker_model(
                **encoded
            )
            .logits
            .view(-1)
        )

        scores = (
            torch.sigmoid(logits)
            .float()
            .cpu()
            .numpy()
        )


    reranked = []

    for candidate, score in zip(
        candidates,
        scores
    ):

        item = dict(candidate)

        item["reranker_score"] = float(
            score
        )

        reranked.append(item)


    reranked.sort(

        key=lambda item:
            item["reranker_score"],

        reverse=True
    )


    return reranked[:final_k]


# ------------------------------------------------------------
# Hybrid retrieval
# ------------------------------------------------------------

def hybrid_retrieve_and_rerank(

    question,
    domain=None,

    dense_k=DENSE_K,
    bm25_k=BM25_K,

    rerank_top_n=RERANK_TOP_N,
    final_k=FINAL_K
):

    domain = (
        normalize_domain(domain)
        if domain
        else None
    )


    # ----------------------------------
    # 1. Dense retrieval using BGE-M3
    # ----------------------------------

    question_embedding = embedder.encode(

        [question],

        normalize_embeddings=True,
        show_progress_bar=False

    ).astype("float32")


    dense_scores, dense_ids = (
        faiss_index.search(

            question_embedding,

            min(
                dense_k,
                faiss_index.ntotal
            )
        )
    )


    dense_ids = (
        dense_ids[0]
        .tolist()
    )


    # ----------------------------------
    # 2. BM25 lexical retrieval
    # ----------------------------------

    question_tokens = tokenize_bn_en(
        question
    )

    bm25_scores = bm25.get_scores(
        question_tokens
    )


    bm25_ids = (

        np.argsort(
            bm25_scores
        )[::-1]

        [:min(
            bm25_k,
            len(chunks)
        )]

        .tolist()
    )


    # ----------------------------------
    # 3. Reciprocal Rank Fusion
    # ----------------------------------

    fused_scores = defaultdict(float)


    for rank, chunk_index in enumerate(

        dense_ids,
        start=1
    ):

        if chunk_index >= 0:

            fused_scores[
                chunk_index
            ] += (

                1.0 /
                (
                    RRF_CONSTANT
                    + rank
                )
            )


    for rank, chunk_index in enumerate(

        bm25_ids,
        start=1
    ):

        fused_scores[
            chunk_index
        ] += (

            1.0 /
            (
                RRF_CONSTANT
                + rank
            )
        )


    # ----------------------------------
    # 4. Domain boost
    # ----------------------------------

    if domain:

        for chunk_index in list(
            fused_scores.keys()
        ):

            chunk_domain = normalize_domain(
                chunks[
                    chunk_index
                ].get(
                    "domain",
                    ""
                )
            )

            if chunk_domain == domain:

                fused_scores[
                    chunk_index
                ] += DOMAIN_BOOST


    ranked = sorted(

        fused_scores.items(),

        key=lambda item:
            item[1],

        reverse=True
    )


    # ----------------------------------
    # 5. Optional known-domain filtering
    # ----------------------------------

    if (
        domain
        and USE_ORACLE_DOMAIN_FILTER
    ):

        domain_ranked = [

            (
                chunk_index,
                score
            )

            for chunk_index, score
            in ranked

            if normalize_domain(
                chunks[
                    chunk_index
                ].get(
                    "domain",
                    ""
                )
            ) == domain
        ]


        if len(domain_ranked) >= final_k:
            ranked = domain_ranked


    # ----------------------------------
    # 6. Prepare candidates
    # ----------------------------------

    candidates = []

    for chunk_index, score in ranked[
        :rerank_top_n
    ]:

        candidate = dict(
            chunks[chunk_index]
        )

        candidate["hybrid_score"] = float(
            score
        )

        candidate[
            "_original_chunk_index"
        ] = int(chunk_index)

        candidates.append(candidate)


    # ----------------------------------
    # 7. Cross-encoder reranking
    # ----------------------------------

    return rerank_candidates(

        question,
        candidates,

        final_k=final_k
    )


# ------------------------------------------------------------
# Format retrieved chunks as model context
# ------------------------------------------------------------

def build_context(
    retrieved_chunks
):

    context_blocks = []

    for rank, chunk in enumerate(

        retrieved_chunks,
        start=1
    ):

        context_blocks.append(

            f"[Context {rank}]\n"

            f"Domain: "
            f"{chunk.get('domain', '')}\n"

            f"Title: "
            f"{chunk.get('title', '')}\n"

            f"Topic: "
            f"{chunk.get('topic', '')}\n"

            f"Source: "
            f"{chunk.get('source_url', '')}\n"

            f"Information:\n"
            f"{get_chunk_text(chunk)}"
        )


    return "\n\n".join(
        context_blocks
    )


# ------------------------------------------------------------
# Quick retrieval test
# ------------------------------------------------------------

sample_question = (
    test_rows[0]["question"]
)

sample_domain = (
    test_rows[0]["domain"]
)


sample_retrieved = (
    hybrid_retrieve_and_rerank(

        question=sample_question,
        domain=sample_domain
    )
)


print("\nSample question:")
print(sample_question)


display(
    pd.DataFrame([

        {
            "rank":
                rank,

            "chunk_id":
                item.get(
                    "chunk_id",
                    item.get(
                        "_original_chunk_index",
                        ""
                    )
                ),

            "domain":
                item.get(
                    "domain",
                    ""
                ),

            "title":
                item.get(
                    "title",
                    ""
                ),

            "hybrid_score":
                item.get(
                    "hybrid_score",
                    0
                ),

            "reranker_score":
                item.get(
                    "reranker_score",
                    0
                )
        }

        for rank, item in enumerate(
            sample_retrieved,
            start=1
        )
    ])
)

Loading embedding model: BAAI/bge-m3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]


Sample question:
ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?


,rank,chunk_id,domain,title,hybrid_score,reranker_score
0,1,chunk_00003,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ,0.057787,1.000000
1,2,chunk_00001,passport,ই-পাসপোর্ট সেবার সারসংক্ষেপ,0.054828,0.991211
2,3,chunk_00004,passport,অনলাইনে ই-পাসপোর্ট আবেদন,0.053373,0.622070
3,4,chunk_00005,passport,ই-পাসপোর্ট ফরম পূরণের সাধারণ নির্দেশনা,0.053283,0.181641
4,5,chunk_00010,passport,পূর্ববর্তী পাসপোর্ট থাকলে করণীয়,0.052746,0.131958


In [10]:
# ============================================================
# CELL 3 — LOAD THE FINE-TUNED QWEN LoRA AND RUN RAG INFERENCE
# Loads the base model written in adapter_config.json, attaches
# the LoRA adapter, retrieves and reranks the top-5 KB chunks,
# generates answers, and saves checkpoints in Google Drive.
# ============================================================

from peft import (
    PeftConfig,
    PeftModel
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)


# ------------------------------------------------------------
# Read base model name from the LoRA adapter
# ------------------------------------------------------------

peft_config = (
    PeftConfig.from_pretrained(
        str(ADAPTER_DIR)
    )
)

BASE_MODEL_NAME = (
    peft_config
    .base_model_name_or_path
)


print("Fine-tuned adapter:")
print(ADAPTER_DIR)

print("\nRequired base model:")
print(BASE_MODEL_NAME)


# ------------------------------------------------------------
# Load tokenizer saved with the adapter
# ------------------------------------------------------------

tokenizer = (
    AutoTokenizer.from_pretrained(

        str(ADAPTER_DIR),

        trust_remote_code=True
    )
)


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


tokenizer.padding_side = "left"


# ------------------------------------------------------------
# Load quantized Qwen base model
# ------------------------------------------------------------

print("\nLoading Qwen base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,

    # Force the complete 4-bit model onto GPU 0
    device_map={"": 0},

    trust_remote_code=True,
    low_cpu_mem_usage=True,

    attn_implementation="sdpa"
)


# ------------------------------------------------------------
# Attach the fine-tuned LoRA adapter
# ------------------------------------------------------------

print("Attaching the LoRA adapter...")


model = PeftModel.from_pretrained(

    base_model,

    str(ADAPTER_DIR),

    is_trainable=False
)

model.eval()


MODEL_DEVICE = (
    model
    .get_input_embeddings()
    .weight
    .device
)


print("\nFine-tuned model loaded successfully.")
print("Model input device:", MODEL_DEVICE)


# ------------------------------------------------------------
# Generate one RAG + fine-tuned answer
# ------------------------------------------------------------

def generate_rag_finetuned_answer(

    question,
    domain,

    top_k=FINAL_K,
    max_new_tokens=180
):

    retrieved_chunks = (
        hybrid_retrieve_and_rerank(

            question=question,
            domain=domain,

            final_k=top_k
        )
    )


    context = build_context(
        retrieved_chunks
    )


    system_prompt = (

        "তুমি বাংলাদেশ সরকারি সেবা বিষয়ক একটি সহায়ক। "

        "শুধুমাত্র দেওয়া Context ব্যবহার করে প্রশ্নের উত্তর দাও। "

        "Context-এর বাইরে কোনো তথ্য তৈরি করবে না। "

        "উত্তর পরিষ্কার, সংক্ষিপ্ত এবং বাংলায় হবে। "

        "প্রয়োজনে ধাপ, প্রয়োজনীয় কাগজপত্র, ফি বা শর্ত "
        "সুস্পষ্টভাবে উল্লেখ করবে। "

        "Context-এ উত্তর না থাকলে বলবে: "
        "'প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।'"
    )


    user_prompt = f"""
Context:
{context}

Question:
{question}

Answer:
""".strip()


    messages = [

        {
            "role": "system",
            "content": system_prompt
        },

        {
            "role": "user",
            "content": user_prompt
        }
    ]


    try:

        prompt = (
            tokenizer.apply_chat_template(

                messages,

                tokenize=False,
                add_generation_prompt=True
            )
        )

    except Exception:

        prompt = (

            f"System:\n{system_prompt}\n\n"

            f"User:\n{user_prompt}\n\n"

            f"Assistant:\n"
        )


    inputs = tokenizer(

        prompt,

        return_tensors="pt",
        truncation=True,

        max_length=8192
    )


    inputs = {

        key: value.to(
            MODEL_DEVICE
        )

        for key, value
        in inputs.items()
    }


    input_length = (
        inputs["input_ids"]
        .shape[1]
    )


    with torch.inference_mode():

        generated_tokens = model.generate(

            **inputs,

            max_new_tokens=max_new_tokens,

            do_sample=False,

            repetition_penalty=1.05,

            pad_token_id=(
                tokenizer.pad_token_id
            ),

            eos_token_id=(
                tokenizer.eos_token_id
            )
        )


    new_tokens = generated_tokens[
        0,
        input_length:
    ]


    answer = tokenizer.decode(

        new_tokens,

        skip_special_tokens=True
    ).strip()


    return answer, retrieved_chunks


# ------------------------------------------------------------
# Output and checkpoint locations
# ------------------------------------------------------------

CHECKPOINT_FILE = (
    ART_DIR /
    "RAG+Fine_tuned_checkpoint.csv"
)

RAW_PREDICTION_FILE = (
    ART_DIR /
    "RAG+Fine_tuned_predictions_raw.csv"
)


RESUME_FROM_CHECKPOINT = True
CHECKPOINT_EVERY = 10


# ------------------------------------------------------------
# Load an existing checkpoint when available
# ------------------------------------------------------------

prediction_rows = []
completed_keys = set()


if (
    RESUME_FROM_CHECKPOINT
    and CHECKPOINT_FILE.exists()
):

    checkpoint_df = pd.read_csv(
        CHECKPOINT_FILE
    )

    prediction_rows = (
        checkpoint_df.to_dict(
            orient="records"
        )
    )


    completed_keys = {

        (
            f"{row['domain']}::"
            f"{normalize_id(row['id'])}"
        )

        for row in prediction_rows
    }


    print(
        f"Resuming from "
        f"{len(prediction_rows)} "
        f"completed questions."
    )


# ------------------------------------------------------------
# Run RAG + fine-tuned inference
# ------------------------------------------------------------

start_time = time.time()


for row in tqdm(

    test_rows,

    desc="RAG + fine-tuned inference"
):

    row_key = (

        f"{row['domain']}::"
        f"{normalize_id(row['id'])}"
    )


    if row_key in completed_keys:
        continue


    generated_answer, retrieved_chunks = (
        generate_rag_finetuned_answer(

            question=row["question"],
            domain=row["domain"],

            top_k=FINAL_K,
            max_new_tokens=180
        )
    )


    result = {

        "id":
            row["id"],

        "domain":
            row["domain"],

        "question":
            row["question"],

        "gold_answer":
            row["gold_answer"],

        "generated_answer":
            generated_answer,

        "gold_source_url":
            row["gold_source_url"]
    }


    # Save the final five reranked chunks.
    for rank in range(
        1,
        FINAL_K + 1
    ):

        if rank <= len(
            retrieved_chunks
        ):

            item = retrieved_chunks[
                rank - 1
            ]

        else:
            item = {}


        result[
            f"retrieved_chunk_id_{rank}"
        ] = item.get(

            "chunk_id",

            item.get(
                "_original_chunk_index",
                ""
            )
        )


        result[
            f"retrieved_title_{rank}"
        ] = item.get(
            "title",
            ""
        )


        result[
            f"retrieved_source_{rank}"
        ] = item.get(
            "source_url",
            ""
        )


        result[
            f"hybrid_score_{rank}"
        ] = item.get(
            "hybrid_score",
            np.nan
        )


        result[
            f"reranker_score_{rank}"
        ] = item.get(
            "reranker_score",
            np.nan
        )


    prediction_rows.append(
        result
    )

    completed_keys.add(
        row_key
    )


    # Save progress every ten questions.
    if (
        len(prediction_rows)
        % CHECKPOINT_EVERY
        == 0
    ):

        pd.DataFrame(
            prediction_rows
        ).to_csv(

            CHECKPOINT_FILE,

            index=False,
            encoding="utf-8-sig"
        )


# ------------------------------------------------------------
# Arrange predictions in original test order
# ------------------------------------------------------------

raw_predictions = pd.DataFrame(
    prediction_rows
)


test_order = {

    (
        f"{row['domain']}::"
        f"{normalize_id(row['id'])}"
    ): index

    for index, row in enumerate(
        test_rows
    )
}


raw_predictions["_order"] = (
    raw_predictions.apply(

        lambda row:

            test_order.get(

                (
                    f"{row['domain']}::"
                    f"{normalize_id(row['id'])}"
                ),

                10**9
            ),

        axis=1
    )
)


raw_predictions = (

    raw_predictions

    .sort_values("_order")

    .drop(
        columns="_order"
    )

    .reset_index(
        drop=True
    )
)


if len(raw_predictions) != len(test_rows):

    raise RuntimeError(

        f"Expected {len(test_rows)} predictions, "

        f"but found {len(raw_predictions)}."
    )


# ------------------------------------------------------------
# Save raw predictions
# ------------------------------------------------------------

raw_predictions.to_csv(

    RAW_PREDICTION_FILE,

    index=False,
    encoding="utf-8-sig"
)


elapsed_minutes = (
    time.time() - start_time
) / 60


print("\nInference completed.")

print(
    "Total predictions:",
    len(raw_predictions)
)

print(
    "Elapsed minutes:",
    round(elapsed_minutes, 2)
)

print(
    "Saved raw predictions:",
    RAW_PREDICTION_FILE
)


display(
    raw_predictions[
        [
            "domain",
            "question",
            "gold_answer",
            "generated_answer",
            "retrieved_title_1",
            "reranker_score_1"
        ]
    ].head(3)
)

Fine-tuned adapter:
/content/Govt_chatbot_finetuned_model_extracted

Required base model:
unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit

Loading Qwen base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Attaching the LoRA adapter...

Fine-tuned model loaded successfully.
Model input device: cuda:0


RAG + fine-tuned inference:   0%|          | 0/248 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

In [9]:
# PATCH — move retrieval models to CPU and free GPU

import gc
import torch

if "base_model" in globals():
    del base_model

if "model" in globals():
    del model

embedder.to("cpu")
reranker_model.to("cpu")

# Cell 2 retrieval functions will now run on CPU
DEVICE = "cpu"

gc.collect()
torch.cuda.empty_cache()

print(
    "Free GPU memory:",
    round(
        torch.cuda.mem_get_info()[0] / 1024**3,
        2
    ),
    "GB"
)

Free GPU memory: 14.41 GB


In [7]:
# ============================================================
# PATCH CELL — REMOVE INCOMPATIBLE TORCHAO
# Run this once before rerunning Cell 3.
#
# The fine-tuned model uses bitsandbytes quantization, not
# TorchAO, so removing TorchAO will not affect this experiment.
# ============================================================

import gc
import sys
import importlib

import torch


# ------------------------------------------------------------
# 1. Delete any partially loaded model from the failed Cell 3
# ------------------------------------------------------------

variables_to_remove = [
    "model",
    "base_model",
    "tokenizer",
    "peft_config",
    "PeftModel",
    "PeftConfig"
]

for variable_name in variables_to_remove:

    if variable_name in globals():

        del globals()[variable_name]


gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


print("Cleared partially loaded model objects.")


# ------------------------------------------------------------
# 2. Remove the incompatible TorchAO installation
# ------------------------------------------------------------

%pip uninstall -y torchao


# ------------------------------------------------------------
# 3. Remove previously cached PEFT and TorchAO modules
# ------------------------------------------------------------

modules_to_remove = [

    module_name

    for module_name in list(sys.modules)

    if (
        module_name == "peft"
        or module_name.startswith("peft.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    )
]


for module_name in modules_to_remove:

    sys.modules.pop(
        module_name,
        None
    )


importlib.invalidate_caches()


# ------------------------------------------------------------
# 4. Verify the patch
# ------------------------------------------------------------

try:

    import importlib.metadata as metadata

    torchao_version = metadata.version(
        "torchao"
    )

    print(
        "TorchAO is still installed:",
        torchao_version
    )

except metadata.PackageNotFoundError:

    print(
        "TorchAO successfully removed."
    )


import peft

print(
    "PEFT version:",
    peft.__version__
)


print(
    "\nPATCH COMPLETED SUCCESSFULLY."
)

print(
    "Now rerun Cell 3 from the beginning."
)

Cleared partially loaded model objects.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
TorchAO successfully removed.
PEFT version: 0.20.0

PATCH COMPLETED SUCCESSFULLY.
Now rerun Cell 3 from the beginning.


In [11]:
# ============================================================
# SPEED PATCH — KEEP QWEN + RERANKER ON GPU
# Keep the embedding model on CPU to reduce GPU-memory pressure.
# ============================================================

import gc
import torch

RERANK_TOP_N = 8
FINAL_K = 5

EMBED_DEVICE = "cpu"
RERANK_DEVICE = "cuda"

embedder.to(EMBED_DEVICE)

reranker_model.to(RERANK_DEVICE)
reranker_model.half()
reranker_model.eval()

gc.collect()
torch.cuda.empty_cache()


def rerank_candidates(question, candidates, final_k=FINAL_K):

    if not candidates:
        return []

    encoded = reranker_tokenizer(
        [question] * len(candidates),
        [get_chunk_text(item) for item in candidates],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    encoded = {
        key: value.to(RERANK_DEVICE)
        for key, value in encoded.items()
    }

    with torch.inference_mode():

        logits = reranker_model(
            **encoded
        ).logits.view(-1)

        scores = torch.sigmoid(
            logits
        ).float().cpu().numpy()

    results = []

    for candidate, score in zip(
        candidates,
        scores
    ):
        item = dict(candidate)
        item["reranker_score"] = float(score)
        results.append(item)

    results.sort(
        key=lambda item: item["reranker_score"],
        reverse=True
    )

    return results[:final_k]


print("Speed patch applied.")
print("Embedder device: CPU")
print("Reranker device: GPU")
print("Rerank candidates:", RERANK_TOP_N)

Speed patch applied.
Embedder device: CPU
Reranker device: GPU
Rerank candidates: 8


In [12]:
# ============================================================
# CELL 3 — REMAINING INFERENCE SECTION
# Uses the already-loaded fine-tuned model and patched fast
# retrieval/reranking functions. Saves progress every 10 rows.
# ============================================================

CHECKPOINT_FILE = ART_DIR / "RAG+Fine_tuned_checkpoint.csv"
RAW_PREDICTION_FILE = ART_DIR / "RAG+Fine_tuned_predictions_raw.csv"

CHECKPOINT_EVERY = 10
MAX_NEW_TOKENS = 180
START_FRESH = False


# ------------------------------------------------------------
# Generate answer using retrieved and reranked context
# ------------------------------------------------------------

def generate_rag_finetuned_answer(question, domain):

    retrieved = hybrid_retrieve_and_rerank(
        question=question,
        domain=domain,
        rerank_top_n=RERANK_TOP_N,
        final_k=FINAL_K
    )

    context = build_context(retrieved)

    messages = [
        {
            "role": "system",
            "content": (
                "তুমি বাংলাদেশ সরকারি সেবা বিষয়ক একটি সহায়ক। "
                "শুধুমাত্র দেওয়া Context ব্যবহার করে উত্তর দাও। "
                "উত্তর পরিষ্কার, সংক্ষিপ্ত এবং বাংলায় হবে। "
                "Context-এ উত্তর না থাকলে বলবে: "
                "'প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।'"
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question:\n{question}\n\n"
                "Answer:"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    )

    inputs = {
        key: value.to(MODEL_DEVICE)
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        output[0][input_length:],
        skip_special_tokens=True
    ).strip()

    return answer, retrieved


# ------------------------------------------------------------
# Start fresh or resume
# ------------------------------------------------------------

if START_FRESH:

    if CHECKPOINT_FILE.exists():
        CHECKPOINT_FILE.unlink()

    if RAW_PREDICTION_FILE.exists():
        RAW_PREDICTION_FILE.unlink()


prediction_rows = []
completed_keys = set()


if CHECKPOINT_FILE.exists():

    checkpoint_df = pd.read_csv(CHECKPOINT_FILE)

    prediction_rows = checkpoint_df.to_dict(
        orient="records"
    )

    completed_keys = {
        f"{row['domain']}::{normalize_id(row['id'])}"
        for row in prediction_rows
    }

    print(
        f"Resuming from {len(prediction_rows)} completed questions."
    )

else:

    print("Starting inference from the beginning.")


# ------------------------------------------------------------
# Run inference
# ------------------------------------------------------------

start_time = time.time()


for row in tqdm(
    test_rows,
    desc="RAG + fine-tuned inference"
):

    row_key = (
        f"{row['domain']}::"
        f"{normalize_id(row['id'])}"
    )

    if row_key in completed_keys:
        continue


    answer, retrieved = generate_rag_finetuned_answer(
        question=row["question"],
        domain=row["domain"]
    )


    result = {
        "id": row["id"],
        "domain": row["domain"],
        "question": row["question"],
        "gold_answer": row["gold_answer"],
        "generated_answer": answer,
        "gold_source_url": row["gold_source_url"]
    }


    for rank in range(1, FINAL_K + 1):

        item = (
            retrieved[rank - 1]
            if rank <= len(retrieved)
            else {}
        )

        result[f"retrieved_chunk_id_{rank}"] = item.get(
            "chunk_id",
            item.get("_original_chunk_index", "")
        )

        result[f"retrieved_title_{rank}"] = item.get(
            "title",
            ""
        )

        result[f"retrieved_source_{rank}"] = item.get(
            "source_url",
            ""
        )

        result[f"hybrid_score_{rank}"] = item.get(
            "hybrid_score",
            np.nan
        )

        result[f"reranker_score_{rank}"] = item.get(
            "reranker_score",
            np.nan
        )


    prediction_rows.append(result)
    completed_keys.add(row_key)


    if len(prediction_rows) % CHECKPOINT_EVERY == 0:

        pd.DataFrame(prediction_rows).to_csv(
            CHECKPOINT_FILE,
            index=False,
            encoding="utf-8-sig"
        )


# ------------------------------------------------------------
# Save completed raw predictions
# ------------------------------------------------------------

raw_predictions = pd.DataFrame(
    prediction_rows
)


test_order = {
    f"{row['domain']}::{normalize_id(row['id'])}": index
    for index, row in enumerate(test_rows)
}


raw_predictions["_order"] = raw_predictions.apply(
    lambda row: test_order.get(
        f"{row['domain']}::{normalize_id(row['id'])}",
        10**9
    ),
    axis=1
)


raw_predictions = (
    raw_predictions
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)


raw_predictions.to_csv(
    RAW_PREDICTION_FILE,
    index=False,
    encoding="utf-8-sig"
)


elapsed_minutes = (
    time.time() - start_time
) / 60


print("\nInference completed.")
print("Predictions:", len(raw_predictions))
print("Elapsed minutes:", round(elapsed_minutes, 2))
print("Saved:", RAW_PREDICTION_FILE)


display(
    raw_predictions[
        [
            "domain",
            "question",
            "gold_answer",
            "generated_answer",
            "retrieved_title_1"
        ]
    ].head(3)
)

Starting inference from the beginning.


RAG + fine-tuned inference:   0%|          | 0/248 [00:00<?, ?it/s]


Inference completed.
Predictions: 248
Elapsed minutes: 123.48
Saved: /content/drive/MyDrive/govt_rag_artifacts/RAG+Fine_tuned_predictions_raw.csv


,domain,question,gold_answer,generated_answer,retrieved_title_1
0,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ হলো: প্রথমে ...,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ
1,passport,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ও ৫ বছর মেয়াদী ই-প...,বিদেশস্থ বাংলাদেশ মিশনের সাধারণ আবেদনকারীদের ই...
2,passport,বিদেশে শ্রমিক শিক্ষার্থীর ৬৪ পৃষ্ঠা ৫ বছর এক্স...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,বিদেশস্থ শ্রমিক ও শিক্ষার্থীদের জন্য ৬৪ পৃষ্ঠা...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...


In [13]:
# ============================================================
# CELL 4 — CALCULATE METRICS AND SAVE FINAL CSV FILES
# Creates:
# 1. RAG+Fine_tuned_predictions.csv
# 2. RAG+Fine_tuned_results.csv
#
# Metrics:
# Normalized Exact Match, Token F1, Fuzzy Match, Corpus BLEU,
# ROUGE-1, ROUGE-2, ROUGE-L, METEOR and BERTScore.
# ============================================================

import nltk

from bert_score import score as bert_score
from nltk.translate.meteor_score import meteor_score
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU


# ------------------------------------------------------------
# Input and output paths
# ------------------------------------------------------------

RAW_PREDICTION_FILE = (
    ART_DIR /
    "RAG+Fine_tuned_predictions_raw.csv"
)

FINAL_PREDICTION_FILE = (
    ART_DIR /
    "RAG+Fine_tuned_predictions.csv"
)

FINAL_RESULTS_FILE = (
    ART_DIR /
    "RAG+Fine_tuned_results.csv"
)


if not RAW_PREDICTION_FILE.exists():

    raise FileNotFoundError(

        "Run Cell 3 first.\n"

        f"Missing file:\n"
        f"{RAW_PREDICTION_FILE}"
    )


df = pd.read_csv(
    RAW_PREDICTION_FILE
)


for column in [

    "id",
    "domain",
    "question",
    "gold_answer",
    "generated_answer"

]:

    df[column] = (

        df[column]
        .fillna("")
        .astype(str)
    )


print(
    "Loaded predictions:",
    len(df)
)


# ------------------------------------------------------------
# Free model memory before loading BERTScore
# ------------------------------------------------------------

for object_name in [

    "model",
    "base_model",
    "reranker_model",
    "embedder"

]:

    if object_name in globals():

        del globals()[
            object_name
        ]


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Bangla-English text normalization
# ------------------------------------------------------------

BANGLA_TO_ENGLISH_DIGITS = (
    str.maketrans(

        "০১২৩৪৫৬৭৮৯",
        "0123456789"
    )
)


def normalize_text(text):

    text = unicodedata.normalize(

        "NFKC",
        str(text)
    )


    text = text.translate(
        BANGLA_TO_ENGLISH_DIGITS
    )


    text = text.lower()


    text = re.sub(

        r"[^\u0980-\u09FFA-Za-z0-9]+",

        " ",

        text
    )


    return re.sub(

        r"\s+",

        " ",

        text
    ).strip()


def metric_tokens(text):

    normalized = normalize_text(
        text
    )

    if not normalized:
        return []

    return normalized.split()


# ------------------------------------------------------------
# Normalized Exact Match
# ------------------------------------------------------------

def normalized_exact_match(
    prediction,
    reference
):

    return int(

        normalize_text(prediction)
        ==
        normalize_text(reference)
    )


# ------------------------------------------------------------
# Token F1
# ------------------------------------------------------------

def token_f1(
    prediction,
    reference
):

    prediction_tokens = metric_tokens(
        prediction
    )

    reference_tokens = metric_tokens(
        reference
    )


    if (
        not prediction_tokens
        and not reference_tokens
    ):
        return 1.0


    if (
        not prediction_tokens
        or not reference_tokens
    ):
        return 0.0


    prediction_counter = Counter(
        prediction_tokens
    )

    reference_counter = Counter(
        reference_tokens
    )


    common_count = sum(

        (
            prediction_counter
            &
            reference_counter
        ).values()
    )


    if common_count == 0:
        return 0.0


    precision = (

        common_count
        /
        len(prediction_tokens)
    )


    recall = (

        common_count
        /
        len(reference_tokens)
    )


    return (

        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# Fuzzy Match
# ------------------------------------------------------------

def fuzzy_match(
    prediction,
    reference
):

    return (

        fuzz.token_set_ratio(

            normalize_text(prediction),

            normalize_text(reference)
        )

        / 100.0
    )


# ------------------------------------------------------------
# ROUGE-1 and ROUGE-2
# ------------------------------------------------------------

def rouge_n_f1(
    prediction,
    reference,
    n
):

    prediction_tokens = metric_tokens(
        prediction
    )

    reference_tokens = metric_tokens(
        reference
    )


    if (
        len(prediction_tokens) < n
        or len(reference_tokens) < n
    ):
        return 0.0


    prediction_ngrams = Counter(

        tuple(
            prediction_tokens[
                index:index + n
            ]
        )

        for index in range(

            len(prediction_tokens)
            - n
            + 1
        )
    )


    reference_ngrams = Counter(

        tuple(
            reference_tokens[
                index:index + n
            ]
        )

        for index in range(

            len(reference_tokens)
            - n
            + 1
        )
    )


    overlap = sum(

        (
            prediction_ngrams
            &
            reference_ngrams
        ).values()
    )


    if overlap == 0:
        return 0.0


    precision = (

        overlap
        /
        sum(
            prediction_ngrams.values()
        )
    )


    recall = (

        overlap
        /
        sum(
            reference_ngrams.values()
        )
    )


    return (

        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# ROUGE-L
# ------------------------------------------------------------

def lcs_length(
    sequence_1,
    sequence_2
):

    previous = [

        0

        for _ in range(
            len(sequence_2) + 1
        )
    ]


    for token_1 in sequence_1:

        current = [0]


        for index, token_2 in enumerate(

            sequence_2,
            start=1
        ):

            if token_1 == token_2:

                current.append(

                    previous[
                        index - 1
                    ] + 1
                )

            else:

                current.append(

                    max(

                        previous[index],

                        current[
                            index - 1
                        ]
                    )
                )


        previous = current


    return previous[-1]


def rouge_l_f1(
    prediction,
    reference
):

    prediction_tokens = metric_tokens(
        prediction
    )

    reference_tokens = metric_tokens(
        reference
    )


    if (
        not prediction_tokens
        or not reference_tokens
    ):
        return 0.0


    longest_common_subsequence = (
        lcs_length(

            prediction_tokens,
            reference_tokens
        )
    )


    precision = (

        longest_common_subsequence
        /
        len(prediction_tokens)
    )


    recall = (

        longest_common_subsequence
        /
        len(reference_tokens)
    )


    if precision + recall == 0:
        return 0.0


    return (

        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# METEOR
# ------------------------------------------------------------

nltk.download(
    "wordnet",
    quiet=True
)

nltk.download(
    "omw-1.4",
    quiet=True
)


def meteor_value(
    prediction,
    reference
):

    prediction_tokens = metric_tokens(
        prediction
    )

    reference_tokens = metric_tokens(
        reference
    )


    if (
        not prediction_tokens
        or not reference_tokens
    ):
        return 0.0


    return float(

        meteor_score(

            [reference_tokens],

            prediction_tokens
        )
    )


# ------------------------------------------------------------
# Calculate per-answer lexical metrics
# ------------------------------------------------------------

pairs = list(

    zip(

        df["generated_answer"],

        df["gold_answer"]
    )
)


df[
    "Normalized Exact Match"
] = [

    normalized_exact_match(
        prediction,
        reference
    )

    for prediction, reference
    in pairs
]


df[
    "Token F1"
] = [

    token_f1(
        prediction,
        reference
    )

    for prediction, reference
    in pairs
]


df[
    "Fuzzy Match"
] = [

    fuzzy_match(
        prediction,
        reference
    )

    for prediction, reference
    in pairs
]


df[
    "ROUGE-1 F1"
] = [

    rouge_n_f1(
        prediction,
        reference,
        n=1
    )

    for prediction, reference
    in pairs
]


df[
    "ROUGE-2 F1"
] = [

    rouge_n_f1(
        prediction,
        reference,
        n=2
    )

    for prediction, reference
    in pairs
]


df[
    "ROUGE-L F1"
] = [

    rouge_l_f1(
        prediction,
        reference
    )

    for prediction, reference
    in pairs
]


df[
    "METEOR"
] = [

    meteor_value(
        prediction,
        reference
    )

    for prediction, reference
    in pairs
]


# ------------------------------------------------------------
# Multilingual BERTScore
# ------------------------------------------------------------

print("\nComputing BERTScore...")


BERT_DEVICE = (

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)


bert_precision, bert_recall, bert_f1 = (

    bert_score(

        df[
            "generated_answer"
        ].tolist(),

        df[
            "gold_answer"
        ].tolist(),

        model_type=(
            "bert-base-multilingual-cased"
        ),

        batch_size=16,

        device=BERT_DEVICE,

        verbose=True,

        idf=False,

        rescale_with_baseline=False
    )
)


df[
    "BERT Precision"
] = (

    bert_precision
    .cpu()
    .numpy()
)


df[
    "BERT Recall"
] = (

    bert_recall
    .cpu()
    .numpy()
)


df[
    "BERT F1"
] = (

    bert_f1
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# Corpus BLEU
# ------------------------------------------------------------

bleu_metric = BLEU(

    tokenize="none",

    smooth_method="exp",

    effective_order=True
)


def calculate_corpus_bleu(
    group
):

    predictions = [

        " ".join(
            metric_tokens(text)
        )

        for text in group[
            "generated_answer"
        ]
    ]


    references = [

        " ".join(
            metric_tokens(text)
        )

        for text in group[
            "gold_answer"
        ]
    ]


    if (
        not any(predictions)
        or not any(references)
    ):
        return 0.0


    bleu_result = (

        bleu_metric.corpus_score(

            predictions,

            [references]
        )
    )


    return (

        bleu_result.score
        /
        100.0
    )


# ------------------------------------------------------------
# Create overall and domain-level result rows
# ------------------------------------------------------------

def create_result_rows(
    group,
    scope,
    domain
):

    metric_scores = {

        "Normalized Exact Match":

            group[
                "Normalized Exact Match"
            ].mean(),


        "Token F1":

            group[
                "Token F1"
            ].mean(),


        "Fuzzy Match":

            group[
                "Fuzzy Match"
            ].mean(),


        "Corpus BLEU":

            calculate_corpus_bleu(
                group
            ),


        "ROUGE-1 F1":

            group[
                "ROUGE-1 F1"
            ].mean(),


        "ROUGE-2 F1":

            group[
                "ROUGE-2 F1"
            ].mean(),


        "ROUGE-L F1":

            group[
                "ROUGE-L F1"
            ].mean(),


        "METEOR":

            group[
                "METEOR"
            ].mean(),


        "BERT Precision":

            group[
                "BERT Precision"
            ].mean(),


        "BERT Recall":

            group[
                "BERT Recall"
            ].mean(),


        "BERT F1":

            group[
                "BERT F1"
            ].mean()
    }


    return [

        {
            "model":
                "Qwen2.5-7B-Instruct + LoRA",

            "setting":
                "Hybrid RAG + reranking + fine-tuned",

            "scope":
                scope,

            "domain":
                domain,

            "count":
                len(group),

            "metric":
                metric,

            "score":
                float(score)
        }

        for metric, score
        in metric_scores.items()
    ]


result_rows = create_result_rows(

    df,

    scope="overall",
    domain="all"
)


for domain, domain_group in df.groupby(

    "domain",
    sort=True
):

    result_rows.extend(

        create_result_rows(

            domain_group,

            scope="domain",
            domain=domain
        )
    )


results = pd.DataFrame(
    result_rows
)


# ------------------------------------------------------------
# Save final requested CSV files
# ------------------------------------------------------------

df.to_csv(

    FINAL_PREDICTION_FILE,

    index=False,

    encoding="utf-8-sig"
)


results.to_csv(

    FINAL_RESULTS_FILE,

    index=False,

    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# Display final results
# ------------------------------------------------------------

print(
    "\n========== "
    "RAG + Fine-tuned Overall Results "
    "=========="
)


display(

    results[

        results["scope"]
        == "overall"

    ][
        [
            "metric",
            "score"
        ]
    ].reset_index(
        drop=True
    )
)


print("\nSaved prediction file:")
print(FINAL_PREDICTION_FILE)

print("\nSaved result file:")
print(FINAL_RESULTS_FILE)

Loaded predictions: 248

Computing BERTScore...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

computing greedy matching.


  0%|          | 0/16 [00:00<?, ?it/s]

done in 1.58 seconds, 156.62 sentences/sec

========== RAG + Fine-tuned Overall Results ==========


,metric,score
0,Normalized Exact Match,0.012097
1,Token F1,0.421754
2,Fuzzy Match,0.698559
3,Corpus BLEU,0.194772
4,ROUGE-1 F1,0.421754
5,ROUGE-2 F1,0.260488
6,ROUGE-L F1,0.387902
7,METEOR,0.364816
8,BERT Precision,0.813302
9,BERT Recall,0.791022



Saved prediction file:
/content/drive/MyDrive/govt_rag_artifacts/RAG+Fine_tuned_predictions.csv

Saved result file:
/content/drive/MyDrive/govt_rag_artifacts/RAG+Fine_tuned_results.csv
